# 🛡️ Zero-Trust Remote LLM & VLM Inference Gateway (`v0.2.0`)

Wrap any OpenAI-compatible local inference engine (`llama-server` / `vLLM` / `Ollama` bound strictly to `127.0.0.1:8080`) behind an **application-layer AES-256-GCM AEAD gateway** on shared or untrusted cloud GPUs (RunPod, Vast.ai, Lambda, AutoDL, or SSH port-forwarded Jupyter pods).

- **Zero Edge Plaintext**: Prompts, system instructions, multimodal images, and streaming SSE responses are sealed end-to-end between your workstation (`127.0.0.1:18443`) and GPU memory.
- **Domain Separation & Anti-Replay**: Uses domain-separated AADs, 24h ticket TTL enforcement, random dynamic routing (`/v1/ks-<hex>`), bounded request-ID `ReplayGuard`, upstream path SSRF allowlisting, and automatic IPython SQLite history scrubbing.

In [ ]:
# === Cell 1: Dependencies, History Scrubbing & In-Memory Passphrase Unlock ===
import subprocess, sys, os, json, sqlite3, time, secrets, getpass
from collections import deque

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

def scrub_ipython_history() -> None:
    try:
        ip = get_ipython()  # noqa: F821
        if ip and getattr(ip, "history_manager", None):
            hm = ip.history_manager
            hm.enabled = False
            if getattr(hm, "db", None):
                try:
                    hm.db.execute("DELETE FROM history")
                    hm.db.commit()
                except Exception:
                    pass
    except Exception:
        pass
    hist_db = os.path.expanduser("~/.ipython/profile_default/history.sqlite")
    if os.path.exists(hist_db):
        try:
            con = sqlite3.connect(hist_db)
            con.execute("DELETE FROM history")
            con.commit()
            con.execute("VACUUM")
            con.close()
        except Exception:
            pass

scrub_ipython_history()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "cryptography", "huggingface_hub"])

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

KDF_ITERATIONS = 210_000
SALT_BYTES, NONCE_BYTES, TAG_BYTES = 16, 12, 16
AAD_PAYLOAD = b"kissshot-payload-v2"
AAD_TICKET = b"kissshot-ticket-v2"
MAX_TTL_SECONDS = 86_400

def derive_key(pwd: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=KDF_ITERATIONS)
    return kdf.derive(pwd.encode("utf-8"))

def encrypt_payload_hex(data: dict, pwd: str, aad: bytes = AAD_PAYLOAD) -> str:
    salt, nonce = secrets.token_bytes(SALT_BYTES), secrets.token_bytes(NONCE_BYTES)
    wrapped = {"_ks_v": 2, "_ks_iat": int(time.time()), "d": data}
    pt = json.dumps(wrapped, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ct = AESGCM(derive_key(pwd, salt)).encrypt(nonce, pt, aad)
    return (salt + nonce + ct).hex()

def decrypt_payload_hex(hex_str: str, pwd: str, aad: bytes = AAD_PAYLOAD, max_age_sec: int = MAX_TTL_SECONDS) -> dict:
    raw = bytes.fromhex("".join(hex_str.split()))
    if len(raw) <= SALT_BYTES + NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated ciphertext")
    pt = AESGCM(derive_key(pwd, raw[:SALT_BYTES])).decrypt(
        raw[SALT_BYTES:SALT_BYTES + NONCE_BYTES],
        raw[SALT_BYTES + NONCE_BYTES:],
        aad,
    )
    parsed = json.loads(pt.decode("utf-8"))
    if isinstance(parsed, dict) and parsed.get("_ks_v") == 2 and "d" in parsed:
        iat = int(parsed.get("_ks_iat", 0) or 0)
        if not iat or abs(time.time() - iat) > max_age_sec:
            raise ValueError("Expired payload")
        return parsed["d"]
    return parsed

def aead_seal(cipher: AESGCM, pt: bytes, aad: bytes) -> bytes:
    nonce = secrets.token_bytes(NONCE_BYTES)
    return nonce + cipher.encrypt(nonce, pt, aad)

def aead_open(cipher: AESGCM, wire: bytes, aad: bytes) -> bytes:
    if len(wire) < NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated frame")
    return cipher.decrypt(wire[:NONCE_BYTES], wire[NONCE_BYTES:], aad)

SESSION_PASSPHRASE = getpass.getpass("🔑 Enter your Kiss-Shot passphrase (in-memory only): ").strip()
assert SESSION_PASSPHRASE, "Passphrase cannot be empty."
print("✅ Passphrase unlocked in memory (IPython history persistence disabled).")

In [ ]:
# === Cell 2: Start Hardened AEAD Gateway in Front of Local Inference Engine (127.0.0.1:8080) ===
# Set INGRESS_HTTPS_URL to your GPU pod's HTTPS proxy URL (e.g. https://<pod>-8081.proxy.runpod.net)
# or http://127.0.0.1:8081 when using SSH local port forwarding (-L 8081:127.0.0.1:8081).
INGRESS_HTTPS_URL = "https://your-pod-8081.example.com"
UPSTREAM_ENGINE_URL = "http://127.0.0.1:8080"
GATEWAY_LISTEN_HOST = "0.0.0.0"
GATEWAY_LISTEN_PORT = 8081

from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import threading, urllib.request, urllib.error

ROUTE_PATH = f"/v1/ks-{secrets.token_hex(8)}"
SESSION_SALT = secrets.token_bytes(SALT_BYTES)
SESSION_CIPHER = AESGCM(derive_key(SESSION_PASSPHRASE, SESSION_SALT))
CONCURRENCY_SEM = threading.BoundedSemaphore(16)
ALLOWED_UPSTREAM_PATHS = {
    "/health",
    "/models",
    "/v1/models",
    "/v1/chat/completions",
    "/v1/completions",
    "/v1/embeddings",
    "/tokenize",
    "/detokenize",
}
_SEEN_IDS, _ID_QUEUE, _LOCK = set(), deque(), threading.Lock()

def _check_replay(req_id: str) -> bool:
    if not req_id or len(req_id) < 8 or len(req_id) > 64:
        return False
    with _LOCK:
        if req_id in _SEEN_IDS:
            return False
        if len(_ID_QUEUE) >= 4096:
            _SEEN_IDS.discard(_ID_QUEUE.popleft())
        _ID_QUEUE.append(req_id)
        _SEEN_IDS.add(req_id)
        return True

class NotebookAEADGateway(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    timeout = 15

    def log_message(self, fmt, *args):
        return

    def _deny(self, status: int = 403):
        raw = b'{"error":"forbidden"}'
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(raw)))
        self.send_header("Connection", "close")
        self.end_headers()
        self.wfile.write(raw)

    def do_GET(self):
        self._deny()

    def do_HEAD(self):
        self._deny()

    def do_POST(self):
        if self.path != ROUTE_PATH:
            self._deny()
            return
        if not CONCURRENCY_SEM.acquire(timeout=3.0):
            self._deny(503)
            return
        try:
            try:
                n = int(self.headers.get("Content-Length", "0") or "0")
                if n <= 0 or n > 64 * 1024 * 1024:
                    self._deny()
                    return
                wire = self.rfile.read(n)
                plain = aead_open(SESSION_CIPHER, wire, f"kissshot-c2s-v1:{ROUTE_PATH}".encode("utf-8"))
                meta_b, req_body = plain.split(b"\n", 1)
                meta = json.loads(meta_b.decode("utf-8"))
                req_id = str(meta.get("id", ""))
                method = str(meta.get("method", "POST")).upper()
                clean_path = str(meta.get("path", "")).split("?", 1)[0].rstrip("/") or "/"
                if (
                    abs(time.time() - int(meta.get("ts", 0))) > 300
                    or not _check_replay(req_id)
                    or method not in ("GET", "POST")
                    or clean_path not in ALLOWED_UPSTREAM_PATHS
                ):
                    self._deny()
                    return
            except Exception:
                self._deny()
                return

            up_req = urllib.request.Request(
                f"{UPSTREAM_ENGINE_URL.rstrip('/')}{clean_path}",
                data=(req_body if method == "POST" else None),
                headers={"Content-Type": "application/json"} if req_body else {},
                method=method,
            )
            try:
                with urllib.request.urlopen(up_req, timeout=300) as resp:
                    ct = (resp.headers.get("Content-Type") or "").lower()
                    if "text/event-stream" in ct:
                        self.send_response(200)
                        self.send_header("Content-Type", "text/event-stream; charset=utf-8")
                        self.send_header("Cache-Control", "no-cache")
                        self.send_header("Connection", "close")
                        self.end_headers()
                        seq = 0
                        while True:
                            ln = resp.readline(4 * 1024 * 1024)
                            if not ln:
                                break
                            enc = aead_seal(SESSION_CIPHER, ln, f"kissshot-s2c:{req_id}:{seq}".encode("ascii")).hex()
                            seq += 1
                            self.wfile.write(f"data: {enc}\n\n".encode("ascii"))
                            self.wfile.flush()
                        eof = aead_seal(SESSION_CIPHER, b"", f"kissshot-eof:{req_id}:{seq}".encode("ascii")).hex()
                        self.wfile.write(f"data: EOF:{eof}\n\n".encode("ascii"))
                        self.wfile.flush()
                    else:
                        out_b = resp.read(32 * 1024 * 1024)
                        hdr_b = json.dumps({"status": resp.status, "ct": "application/json"}).encode("utf-8")
                        enc_all = aead_seal(SESSION_CIPHER, hdr_b + b"\n" + out_b, f"kissshot-s2c-body:{req_id}".encode("ascii"))
                        self.send_response(200)
                        self.send_header("Content-Type", "application/octet-stream")
                        self.send_header("Content-Length", str(len(enc_all)))
                        self.send_header("Connection", "close")
                        self.end_headers()
                        self.wfile.write(enc_all)
            except urllib.error.HTTPError as he:
                err_b = he.read(4 * 1024 * 1024)
                hdr_b = json.dumps({"status": he.code, "ct": "application/json"}).encode("utf-8")
                enc_err = aead_seal(SESSION_CIPHER, hdr_b + b"\n" + err_b, f"kissshot-s2c-body:{req_id}".encode("ascii"))
                self.send_response(200)
                self.send_header("Content-Type", "application/octet-stream")
                self.send_header("Content-Length", str(len(enc_err)))
                self.send_header("Connection", "close")
                self.end_headers()
                self.wfile.write(enc_err)
        finally:
            CONCURRENCY_SEM.release()

ticket_payload = {
    "v": 2,
    "type": "kissshot_handshake",
    "url": INGRESS_HTTPS_URL.strip().rstrip("/"),
    "route": ROUTE_PATH,
    "salt": SESSION_SALT.hex(),
    "ts": int(time.time()),
}
ticket_line = "KISSSHOT_TICKET:" + encrypt_payload_hex(ticket_payload, SESSION_PASSPHRASE, aad=AAD_TICKET)

gw = ThreadingHTTPServer((GATEWAY_LISTEN_HOST, GATEWAY_LISTEN_PORT), NotebookAEADGateway)
print("=" * 76)
print("🎯 Remote AEAD Gateway Ready! Copy the KISSSHOT_TICKET line below to your workstation:\n")
print(ticket_line)
print("\n" + "=" * 76)
try:
    gw.serve_forever()
except KeyboardInterrupt:
    gw.shutdown()